# Scikit Feature Tracking Algorithm

### Import Packages

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from skimage.measure import label, regionprops
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point
from matplotlib.ticker import MaxNLocator, FuncFormatter

## Let's go through the basic steps to identify SLP features together with precipitation

Open the dataset using xarray

In [ ]:
ds_SLP = xr.open_dataset("data/SLP_ex.nc")
ds_precip = xr.open_dataset("data/precip_ex.nc")

Select only Northern Hemisphere since Precipitation data is limited

In [ ]:
da_SLP = ds_SLP.SLP
lat_name = da_SLP.dims[1]
lat_vals = da_SLP[lat_name]
da_SLP = da_SLP.where(lat_vals >= 0, drop=True)
da_precip = ds_precip.ETC_tp

A few parameter for identifying the feature objects and tracking overlaps through time

In [ ]:
threshold = 100500
min_area = 20
min_overlap = 0.2  

Get latitude and longitude coordinate and 2d meshgrid for plotting

In [ ]:
lat_SLP = da_SLP[da_SLP.dims[1]].values
lon_SLP = da_SLP[da_SLP.dims[2]].values

lat_precip = da_precip[da_precip.dims[1]].values
lon_precip = da_precip[da_precip.dims[2]].values

# if lat/lon are 1D, make 2D grids for contourf/contour
lon2d_SLP, lat2d_SLP = np.meshgrid(lon_SLP, lat_SLP)
lon2d_precip, lat2d_precip = np.meshgrid(lon_precip, lat_precip)

## Select low pressure features less than a threshold using Scikit

In [ ]:
def detect_features(field, threshold=threshold, min_area=20):
    mask = field <= threshold   # low pressure: SLP less than threshold

    labels = label(mask, connectivity=2)

    for lab in np.unique(labels)[1:]:
        if np.sum(labels == lab) < min_area:
            labels[labels == lab] = 0

    labels = label(labels > 0, connectivity=2)

    return labels

## Precompute and Track SLP Features Over Time

This block detects SLP features at each time step and gives each feature a track ID so we can follow it from one frame to the next.

In simple terms:

1. Loop through every time step in the SLP data.
2. Detect features in the current SLP field using a threshold and minimum area.
3. For each detected feature, compare it with features from the previous time step.
4. If the new feature overlaps enough with an old feature, it keeps the same track_id.
5. If it does not overlap enough, it is treated as a new feature and gets a new track_id.
6. Store the feature’s ID, centroid location, and mask for plotting or later analysis.

In [ ]:
tracked = []
prev = {}
next_track_id = 1

for t in range(da_SLP.sizes["time"]):
    field = da_SLP.isel(time=t).values
    labels = detect_features(field, threshold=threshold, min_area=min_area)

    current = {}
    frame_info = []

    for r in regionprops(labels):
        mask = labels == r.label

        best_id = None
        best_overlap = 0.0

        for track_id, old_mask in prev.items():
            overlap = np.logical_and(mask, old_mask).sum() / mask.sum()
            if overlap > best_overlap:
                best_overlap = overlap
                best_id = track_id

        if best_overlap < min_overlap:
            best_id = next_track_id
            next_track_id += 1

        current[best_id] = mask
        frame_info.append({
            "track_id": best_id,
            "centroid_rc": r.centroid,   # row, col
            "mask": mask
        })

    tracked.append(frame_info)
    prev = current

## Let's look at the animation of the SLP features with Precip overlayed

In [ ]:
# ---- scales ----
slp_vmin = float(da_SLP.min())
slp_vmax = float(da_SLP.max())
slp_levels = np.linspace(slp_vmin, slp_vmax, 16)
slp_norm = mcolors.Normalize(slp_vmin, slp_vmax)

precip_floor = 0.05
precip_vmax = float(da_precip.quantile(0.99))
precip_norm = mcolors.Normalize(precip_floor, precip_vmax)

precip_cmap = plt.get_cmap("gray").copy()
precip_cmap.set_bad(alpha=0)


# ---- cyclic longitude for SLP filled contour ----
# This prevents a visual seam near 0/360 longitude.
lat2d_SLP_cyc = None

def get_cyclic_slp(slp2d):
    slp_cyc, lon_cyc = add_cyclic_point(slp2d, coord=lon_SLP)
    lon2d_cyc, lat2d_cyc = np.meshgrid(lon_cyc, lat_SLP)
    return slp_cyc, lon2d_cyc, lat2d_cyc


# ---- figure ----
fig, ax = plt.subplots(
    figsize=(10, 7),
    dpi=200,
    subplot_kw={"projection": ccrs.PlateCarree()}
)

fig.subplots_adjust(bottom=0.28)

cbar_slp = fig.colorbar(
    ScalarMappable(norm=slp_norm, cmap="turbo"),
    ax=ax, orientation="horizontal",
    shrink=0.75, pad=0.08, aspect=35
)
cbar_slp.set_label("SLP (hPa)")
cbar_slp.locator = MaxNLocator(nbins=5)
cbar_slp.formatter = FuncFormatter(lambda x, pos: f"{x/100:.0f}")
cbar_slp.update_ticks()

cbar_precip = fig.colorbar(
    ScalarMappable(norm=precip_norm, cmap=precip_cmap),
    ax=ax, orientation="horizontal",
    shrink=0.75, pad=0.18, aspect=35
)
cbar_precip.set_label("Precipitation (mm/h)")
cbar_precip.locator = MaxNLocator(nbins=4)
cbar_precip.update_ticks()


def update(t):
    ax.clear()

    slp = da_SLP.isel(time=t).values
    precip = da_precip.isel(time=t).values
    precip_plot = np.where(precip >= precip_floor, precip, np.nan)

    slp_cyc, lon2d_SLP_cyc, lat2d_SLP_cyc = get_cyclic_slp(slp)

    # ---- SLP filled background ----
    ax.contourf(
        lon2d_SLP_cyc,
        lat2d_SLP_cyc,
        slp_cyc,
        levels=slp_levels,
        cmap="turbo",
        norm=slp_norm,
        extend="both",
        transform=ccrs.PlateCarree(),
        zorder=1
    )

    # ---- precipitation overlay ----
    ax.pcolormesh(
        lon2d_precip,
        lat2d_precip,
        precip_plot,
        cmap=precip_cmap,
        norm=precip_norm,
        shading="auto",
        alpha=0.95,
        transform=ccrs.PlateCarree(),
        zorder=2
    )

    # ---- tracked outlines and labels ----
    for feat in tracked[t]:
        mask = feat["mask"]

        ax.contour(
            lon2d_SLP,
            lat2d_SLP,
            mask.astype(int),
            levels=[0.5],
            colors="black",
            linewidths=0.9,
            alpha=0.6,
            transform=ccrs.PlateCarree(),
            zorder=3
        )

        y, x = feat["centroid_rc"]
        iy = int(round(y))
        ix = int(round(x))

        ax.text(
            lon_SLP[ix],
            lat_SLP[iy],
            str(feat["track_id"]),
            fontsize=7,
            color="black",
            ha="center",
            va="center",
            transform=ccrs.PlateCarree(),
            zorder=4
        )

    ax.coastlines(color="0.4", linewidth=0.5)

    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.3,
        color="0.5",
        alpha=0.25,
        linestyle="--"
    )
    gl.top_labels = False
    gl.right_labels = False

    ax.set_title(f"Scikit tracked SLP features with Precipitation | time index = {t}")


anim = FuncAnimation(
    fig,
    update,
    frames=da_SLP.sizes["time"],
    interval=300,
    blit=False
)

plt.close(fig)
HTML(anim.to_jshtml())

- ERA 5 data
- 2022 November 1
- Ran through each hour
- SLP feature in black lines at 1005hPa
- Precipitation in greyscale low precip in balck going white at high pressure (mm/h)
- SLP drawn in rainbow low to high blue to red (hPa)
- What this plot shows is that some of the precipitation features is enveloped within these vast SLP low pressure objects
- Important to note that with SLP there would be topographic differences between maritime and inland.
- Couple of systems in Northern America and over Europe
- There are systems of precipitation which are not within these defined objects of SLP in subtropics for exaample in Japan and northern hemisphere subtropical atlantic. 